DH演習2026 · 教材 第42版 · 2026-r42 · 2026-10-23

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第42版
教材ID: 2026-r42
教材日付: 2026-10-23
活動: relation_network
環境: Colab / Jupyter
ソース: 62acf1f49a60d16bf906f1bdc0bd345b8ac36503
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第11回：線を引く規則を変えると、誰が中心になるか

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

資料は芥川龍之介『二人小町』（[TEI-EAJ aozora_tei](https://github.com/TEI-EAJ/aozora_tei/blob/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml)、CC0）です。関係表は授業で作った判断案です。

## 1. 本文を読む

ノートブックはTEIを取得し、カードA（第1場の発言14–21）、B（第2場の発言16–20）、C（第3場の発言1–14）をルビの読みを除いて表示します。［ト書き］は話者の発言の外にある舞台指示です。

In [ ]:
#@title 操作：本文を取得して表示する
import xml.etree.ElementTree as ET

import networkx as nx
import requests

url = "https://raw.githubusercontent.com/TEI-EAJ/aozora_tei/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml"
T = "{http://www.tei-c.org/ns/1.0}"
tei = ET.fromstring(requests.get(url).content)
for span in tei.iter(T + "span"):
    if span.get("type") in ("rt", "rp"):
        span.text = ""
cards = {"A": (1, 14, 21), "B": (2, 16, 20), "C": (3, 1, 14)}
for card, (scene, first, last) in cards.items():
    print("カード" + card)
    n = 0
    for e in tei.findall(f".//{T}div")[scene - 1]:
        if e.tag == T + "sp":
            n += 1
            speaker, *rest = e
            if first <= n <= last:
                print(f"  s{scene}-{n} {speaker.text}：" + "".join("".join(x.itertext()) for x in rest))
        elif e.tag == T + "stage" and first <= n + 1 <= last:
            print("  ［ト書き］" + "".join(e.itertext()).strip())

## 2. 関係表を判断する

一つのカードでの一組ごとに、`co_mention`（共起：発言の話者と、その発言本文に書かれたフルネーム）と`interaction`（相互行為：対面の会話か直接の働きかけ）を、`yes`／`no`／`defer`で別々に判断してください。R1・R2は記入例です。R3–R7は本文と照合し、`basis`に箇所と理由を書いてください。

`unreviewed`は点検前の未判断です。点検しても根拠が足りない場合は`defer`（保留）にし、理由を記してください。

In [ ]:
import pandas as pd

relations = pd.DataFrame([
    ("R1", "A", "ono", "tama", "yes", "no", "s1-15で小野が玉造のフルネームを言う。玉造は不在で、会話はない"),
    ("R2", "A", "ono", "messenger", "no", "yes", "発言にフルネームはない。s1-14–20で身代わりを交渉する"),
    ("R3", "B", "messenger", "tama", "unreviewed", "unreviewed", ""),
    ("R4", "B", "messenger", "ono", "unreviewed", "unreviewed", ""),
    ("R5", "B", "ono", "tama", "unreviewed", "unreviewed", ""),
    ("R6", "C", "messenger", "guards", "unreviewed", "unreviewed", ""),
    ("R7", "C", "guards", "ono", "unreviewed", "unreviewed", ""),
], columns=["record_id", "card", "source", "target", "co_mention", "interaction", "basis"])
relations

## 3. 線を引く規則を選ぶ

ノードは四つの人物・集団、線は選んだ列が`yes`の行です。`use_interactions`を`True`にすると、共起の列から相互行為の列へ切り替わります。変える前に、消える線と増える線を予測してください。

In [ ]:
use_interactions = False

column = "interaction" if use_interactions else "co_mention"
g = nx.Graph()
g.add_nodes_from(["ono", "tama", "messenger", "guards"])
for row in relations.itertuples():
    if getattr(row, column) == "yes":
        g.add_edge(row.source, row.target)
print(column, list(g.edges))
nx.draw_networkx(g, pos=nx.circular_layout(g), node_color="lightgray", node_size=1600, margins=0.15)

表示された線のうち一つを選んでください。その線を加えた表の行は、`if`のどの条件を満たしていますか。

## 4. 次数中心性を手で求めて比べる

次数は線でつながる相手の数、次数中心性は次数÷（ノード数−1）です。一人の値を手で求めてから実行してください。

In [ ]:
centrality = nx.degree_centrality(g)
pd.DataFrame({"次数": dict(g.degree), "次数中心性": centrality})

この値は、今回の線の規則とカードの範囲で、線でつながる相手の多さを示します。文学的な重要性と読むには、本文の何を別に確かめる必要があるでしょうか。

## 確認問題

2.の表でR7の`interaction`だけを`yes`にし、3.を`use_interactions = True`で再実行してください。このとき`ono`の次数中心性はいくつですか。`answer`に数で入れてください。

In [ ]:
answer = None

In [ ]:
#@title 答えを確かめる
hints = [
    (2 / 3, ""),
    (1 / 3, "R7がまだ反映されていません"),
    (2, "それは次数です。次数中心性は次数÷（全ノード数−1）です"),
    (0.5, "線のない人物も含めて、ノードは4つです"),
    (1, "R7以外の行も変えていないか確かめてください"),
]
if answer is None:
    print("結果：不合格（`answer`に数を入れてください）")
else:
    hint = next((h for v, h in hints if abs(answer - v) < 0.006), "`ono`と線でつながる相手を数え、全ノード数−1で割ってください")
    print(f"結果：不合格（{hint}）" if hint else "結果：合格")